# Ansatz comparison: ODRA, ODRA-native, simulator

Evaluates all three ansätze trained in [`setup/train_comparable_ansatze.ipynb`](setup/train_comparable_ansatze.ipynb): **forward ring** + **wrap-first** second entangling block, **Z on `q_0`**.

- **Weights:** `weights_odra_final.pth`, `weights_odra_native_final.pth`, `weights_simulator_final.pth` (last epoch).
- **Test CSV:** [`setup/test_data.csv`](setup/test_data.csv) (same split/scaling as that notebook).

- **StatevectorEstimator**: deterministic baseline (std reported as 0).
- **IQM** (optional): set **`RUN_IQM_HARDWARE = True`** for repeated shot evaluation on **IQM Spark ODRA**; otherwise only statevector metrics are computed and IQM columns in the summary are **NaN**.

**Dependencies:** Qiskit 2.x, `qiskit-machine-learning`, `iqm-client[qiskit]` (if using IQM), PyTorch, scikit-learn, pandas. Prefer the project **uv** environment ([`pyproject.toml`](../../pyproject.toml)).

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --- Evaluation configuration ---
NUM_QUBITS = 5
ANSATZ_DEPTH = 2
RANDOM_SEED = 42

# Shots per circuit on IQM (only if RUN_IQM_HARDWARE)
EVAL_SHOTS = 512

# At least 10 repeats per ansatz on hardware when RUN_IQM_HARDWARE is True
N_REPEATS = 125

# Set True to prompt for IQM token and run shot-based evaluation on hardware
RUN_IQM_HARDWARE = False

# Artifacts from setup/train_comparable_ansatze.ipynb
SETUP_DIR = "tests/ansatz_odra_evaluation/setup"
TEST_CSV_NAME = "test_data.csv"
WEIGHTS_ODRA_NAME = "weights_odra_final.pth"
WEIGHTS_ODRA_NATIVE_NAME = "weights_odra_native_final.pth"
WEIGHTS_SIM_NAME = "weights_simulator_final.pth"

# CSV output directory is fixed under the repo: tests/ansatz_odra_evaluation/outputs/run_<timestamp>/

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import PrimitiveResult, PubResult, StatevectorEstimator
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.primitives.containers.data_bin import DataBin
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.neural_networks import EstimatorQNN
from sklearn.metrics import accuracy_score, f1_score

try:
    from tqdm import tqdm

    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False


def _project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "cross_validation").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root (folder with cross_validation/).")


def load_setup_test_data(root: Path, setup_relative: str, csv_name: str) -> tuple[np.ndarray, np.ndarray]:
    path = root / setup_relative / csv_name
    if not path.is_file():
        raise FileNotFoundError(path)
    data = np.loadtxt(path, delimiter=",", skiprows=1, ndmin=2)
    if data.shape[1] != 6:
        raise ValueError(f"Expected 6 columns in {path}, got {data.shape[1]}")
    x = data[:, :5].astype(np.float32)
    y = data[:, 5].astype(np.float32)
    # Setup notebook writes labels in {0, 1}; metrics match fold CSVs using {-1, +1}
    if set(np.unique(y)).issubset({0.0, 1.0}):
        y = 2.0 * y - 1.0
    return x, y


def predictions_to_labels(predictions: np.ndarray) -> np.ndarray:
    predictions = np.asarray(predictions).reshape(-1)
    return np.where(predictions > 0, 1, -1).astype(np.float32)


ROOT = _project_root()
WEIGHTS_ODRA = ROOT / SETUP_DIR / WEIGHTS_ODRA_NAME
WEIGHTS_ODRA_NATIVE = ROOT / SETUP_DIR / WEIGHTS_ODRA_NATIVE_NAME
WEIGHTS_SIM = ROOT / SETUP_DIR / WEIGHTS_SIM_NAME
TEST_DATA_CSV = ROOT / SETUP_DIR / TEST_CSV_NAME

print(f"Project root: {ROOT}")
print(f"Test CSV: {TEST_DATA_CSV} (exists={TEST_DATA_CSV.is_file()})")
print(f"ODRA weights: {WEIGHTS_ODRA} (exists={WEIGHTS_ODRA.is_file()})")
print(f"ODRA-native weights: {WEIGHTS_ODRA_NATIVE} (exists={WEIGHTS_ODRA_NATIVE.is_file()})")
print(f"Simulator weights: {WEIGHTS_SIM} (exists={WEIGHTS_SIM.is_file()})")
print(f"RUN_IQM_HARDWARE={RUN_IQM_HARDWARE}")

In [ ]:
def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """Same as setup/train_comparable_ansatze.ipynb: ring + wrap-first second block."""
    params_per_iter = 4 * n_qubits
    theta = ParameterVector("theta", params_per_iter * (depth // 2))
    qc = QuantumCircuit(n_qubits)

    for j in range(depth // 2):
        offset = j * params_per_iter

        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            param_idx = offset + n_qubits + i
            qc.rz(theta[param_idx], target)
            qc.cz(control, target)

        offset_l2 = offset + 2 * n_qubits
        for i in range(n_qubits):
            qc.rx(theta[offset_l2 + i], i)

        base3 = offset_l2 + n_qubits
        qc.ry(theta[base3 + 0], n_qubits - 1)
        qc.cz(0, n_qubits - 1)
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.ry(theta[base3 + k], target)
            qc.cz(control, target)

    return qc


def odra_native_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """R(θ,φ) + CZ; same topology as odra_ansatz."""
    params_per_iter = 8 * n_qubits
    theta = ParameterVector("theta", params_per_iter * (depth // 2))
    qc = QuantumCircuit(n_qubits)

    for j in range(depth // 2):
        offset = j * params_per_iter
        p = offset

        for i in range(n_qubits):
            qc.r(theta[p], theta[p + 1], i)
            p += 2

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.r(theta[p], theta[p + 1], target)
            p += 2
            qc.cz(control, target)

        for i in range(n_qubits):
            qc.r(theta[p], theta[p + 1], i)
            p += 2

        qc.r(theta[p], theta[p + 1], n_qubits - 1)
        p += 2
        qc.cz(0, n_qubits - 1)
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.r(theta[p], theta[p + 1], target)
            p += 2
            qc.cz(control, target)

    return qc


def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    theta = ParameterVector("theta", 2 * n_qubits * depth)
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for _ in range(depth // 2):
        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        qc.cry(theta[param_idx], 0, n_qubits - 1)
        param_idx += 1
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.cry(theta[param_idx], control, target)
            param_idx += 1

    return qc


_od = odra_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
_nat = odra_native_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
_sim = simulator_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
assert len(list(_od.parameters)) == 4 * NUM_QUBITS * (ANSATZ_DEPTH // 2)
assert len(list(_nat.parameters)) == 8 * NUM_QUBITS * (ANSATZ_DEPTH // 2)
assert len(list(_sim.parameters)) == 2 * NUM_QUBITS * ANSATZ_DEPTH
print(
    f"Ansatz params: odra={len(list(_od.parameters))}, "
    f"odra_native={len(list(_nat.parameters))}, simulator={len(list(_sim.parameters))}"
)

In [ ]:
class SimpleIQMJob:
    def __init__(self, result):
        self._result = result

    def result(self):
        return self._result


class IQMBackendEstimator(BaseEstimatorV2):
    def __init__(self, backend, options=None):
        super().__init__()
        self._backend = backend
        self._options = options or {"shots": 100}
        self.timestamp_history = []
        self.total_qpu_time = 0.0

    def _extract_timestamps(self, result):
        try:
            timeline = result._metadata.get("timeline", [])
            if not timeline:
                return None
            timestamps = {}
            for entry in timeline:
                timestamps[entry.status] = entry.timestamp
            return timestamps
        except Exception:
            return None

    def _counts_to_expectation(self, counts):
        if isinstance(counts, list):
            counts = counts[0]
        shots = sum(counts.values())
        count_0 = sum(c for bitstring, c in counts.items() if bitstring[-1] == "0")
        p0 = count_0 / shots if shots else 0.0
        return p0 - (1 - p0)

    def run(self, pubs, precision=None):
        if not isinstance(pubs, list):
            pubs = [pubs]

        job_results = []
        shots_opt = self._options["shots"]
        max_circuits = self._options.get("max_circuits_per_job")

        base_circuit = pubs[0][0]
        circuit_with_meas = base_circuit.copy()
        if circuit_with_meas.num_clbits == 0:
            circuit_with_meas.measure_all()
        transpiled_qc = transpile(circuit_with_meas, self._backend, optimization_level=3)

        for pub in pubs:
            _, observables, parameter_values = pub
            if parameter_values.ndim == 1:
                parameter_values = [parameter_values]

            bound_circuits = [transpiled_qc.assign_parameters(params) for params in parameter_values]
            n_circuits = len(bound_circuits)
            pub_expectations = []

            for start in range(0, n_circuits, max_circuits or n_circuits):
                end = min(start + (max_circuits or n_circuits), n_circuits)
                batch = bound_circuits[start:end]
                try:
                    job = self._backend.run(batch, shots=shots_opt)
                    result = job.result()

                    ts = self._extract_timestamps(result)
                    if ts:
                        exec_start = ts.get("execution_started")
                        exec_end = ts.get("execution_ended")
                        comp_start = ts.get("compilation_started")
                        comp_end = ts.get("compilation_ended")
                        job_created = ts.get("created")
                        job_completed = ts.get("completed")
                        if exec_start and exec_end:
                            execution_time = (exec_end - exec_start).total_seconds()
                            compile_time = (
                                (comp_end - comp_start).total_seconds() if comp_start and comp_end else 0.0
                            )
                            job_time = (
                                (job_completed - job_created).total_seconds()
                                if job_created and job_completed
                                else 0.0
                            )
                            self.timestamp_history.append(
                                {
                                    "execution_time_qpu": execution_time,
                                    "job_time_total": job_time,
                                    "compile_time": compile_time,
                                    "raw_timestamps": ts,
                                    "n_circuits": len(batch),
                                }
                            )
                            self.total_qpu_time += execution_time

                    counts_list = result.get_counts()
                    if not isinstance(counts_list, list):
                        counts_list = [counts_list]
                    for counts in counts_list:
                        pub_expectations.append(self._counts_to_expectation(counts))
                except Exception as exc:
                    print(f"Batch job failed: {exc}")
                    pub_expectations.extend([0.0] * len(batch))

            data = DataBin(evs=np.array(pub_expectations), shape=(len(pub_expectations),))
            job_results.append(PubResult(data=data))

        return SimpleIQMJob(PrimitiveResult(job_results))

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit: QuantumCircuit, num_qubits: int):
        super().__init__()
        self.feature_map = self.angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        estimator = StatevectorEstimator(seed=RANDOM_SEED)
        gradient = ParamShiftEstimatorGradient(estimator=estimator)

        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient,
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector("x", num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x):
        return self.quantum_layer(x)


def build_statevector_model(ansatz_fn, num_qubits: int, depth: int) -> HybridModel:
    circ = ansatz_fn(num_qubits, depth)
    return HybridModel(circ, num_qubits)


def build_iqm_model(iqm_backend, ansatz_fn, n_shots: int, num_qubits: int, depth: int):
    hw_estimator = IQMBackendEstimator(iqm_backend, options={"shots": n_shots})
    hw_ansatz = ansatz_fn(num_qubits, depth)
    hw_feature_map = HybridModel(hw_ansatz, num_qubits).angle_encoding(num_qubits)

    hw_qc = QuantumCircuit(num_qubits)
    hw_qc.compose(hw_feature_map, qubits=range(num_qubits), inplace=True)
    hw_qc.compose(hw_ansatz, inplace=True)

    observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
    hw_qnn = EstimatorQNN(
        circuit=hw_qc,
        observables=observable,
        input_params=list(hw_feature_map.parameters),
        weight_params=list(hw_ansatz.parameters),
        estimator=hw_estimator,
    )
    hw_model = TorchConnector(hw_qnn)
    return hw_model, hw_estimator


def _torch_load(path: Path):
    import importlib
    import pickle

    # Always use a fresh binding to the real package (global ``torch`` can be shadowed). Jupyter
    # autoreload sometimes leaves the package without ``._utils`` in ``__dict__``, which breaks
    # ``torch.load`` even with ``weights_only=False``.
    import torch as _tp

    if _tp.__dict__.get("_utils") is None:
        _tp.__dict__["_utils"] = importlib.import_module("torch._utils")

    try:
        return _tp.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return _tp.load(path, map_location="cpu")
    except (AttributeError, pickle.UnpicklingError):
        return _tp.load(path, map_location="cpu", weights_only=False)


def _unwrap_state_dict(obj):
    if isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
        return obj["state_dict"]
    return obj


def _strip_quantum_layer_prefix(state: dict) -> dict:
    out = {}
    for key, value in state.items():
        if key.startswith("quantum_layer."):
            out[key.replace("quantum_layer.", "", 1)] = value
        else:
            out[key] = value
    return out


def load_checkpoint_hybrid(model: HybridModel, path: Path) -> None:
    raw = _unwrap_state_dict(_torch_load(path))
    if not isinstance(raw, dict):
        raise TypeError(f"Expected dict checkpoint at {path}")
    try:
        model.load_state_dict(raw, strict=True)
        return
    except Exception:
        pass
    stripped = _strip_quantum_layer_prefix(raw)
    model.quantum_layer.load_state_dict(stripped, strict=True)


def load_checkpoint_connector(connector: TorchConnector, path: Path) -> None:
    raw = _unwrap_state_dict(_torch_load(path))
    if not isinstance(raw, dict):
        raise TypeError(f"Expected dict checkpoint at {path}")
    stripped = _strip_quantum_layer_prefix(raw)
    connector.load_state_dict(stripped, strict=True)


def connect_to_iqm_backend():
    base_url = "https://odra5.e-science.pl/"
    token = input("Enter IQM Token: ").strip()
    if not token:
        raise ValueError("An IQM token is required.")
    from iqm.qiskit_iqm import IQMProvider

    provider = IQMProvider(base_url, token=token)
    backend = provider.get_backend()
    print(f"Connected to backend: {backend.name}")
    return backend

## Run comparison

1. Run [`setup/train_comparable_ansatze.ipynb`](setup/train_comparable_ansatze.ipynb) so `setup/test_data.csv` and the three `weights_*_final.pth` files exist.
2. Optional: set **`RUN_IQM_HARDWARE = True`**, **`EVAL_SHOTS`**, and **`N_REPEATS`** (≥10) for IQM evaluation (token prompt).
3. With **`RUN_IQM_HARDWARE = False`** (default), only **Statevector** accuracy/F1 are computed for all three ansätze.
4. Results are written to **`tests/ansatz_odra_evaluation/outputs/run_<timestamp>/`** (`run_level_results.csv` is empty if IQM was skipped).

In [ ]:
if RUN_IQM_HARDWARE:
    if N_REPEATS < 10:
        raise ValueError("N_REPEATS must be at least 10 when RUN_IQM_HARDWARE is True.")
    if EVAL_SHOTS <= 0:
        raise ValueError("EVAL_SHOTS must be positive.")

_weight_paths = [WEIGHTS_ODRA, WEIGHTS_ODRA_NATIVE, WEIGHTS_SIM]
if not all(p.is_file() for p in _weight_paths) or not TEST_DATA_CSV.is_file():
    miss = [str(p) for p in _weight_paths + [TEST_DATA_CSV] if not p.is_file()]
    raise FileNotFoundError("Missing setup artifacts (run setup/train_comparable_ansatze.ipynb first): " + ", ".join(miss))

X_test, y_test = load_setup_test_data(ROOT, SETUP_DIR, TEST_CSV_NAME)
X_tensor = torch.tensor(X_test, dtype=torch.float32)
n_samples = len(y_test)

try:
    from IPython.display import display
except ImportError:
    display = print

run_stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S")
out_dir = ROOT / "tests" / "ansatz_odra_evaluation" / "outputs" / f"run_{run_stamp}"
out_dir.mkdir(parents=True, exist_ok=True)

ansatz_specs = [
    ("odra", odra_ansatz, WEIGHTS_ODRA),
    ("odra_native", odra_native_ansatz, WEIGHTS_ODRA_NATIVE),
    ("simulator", simulator_ansatz, WEIGHTS_SIM),
]

run_rows = []
summary_rows = []

print(f"Test data {TEST_CSV_NAME}: {n_samples} samples → outputs: {out_dir}")

iqm_backend = connect_to_iqm_backend() if RUN_IQM_HARDWARE else None

for ansatz_name, ansatz_fn, weight_path in ansatz_specs:
    print(f"\n--- {ansatz_name} ---")

    sv_model = build_statevector_model(ansatz_fn, NUM_QUBITS, ANSATZ_DEPTH)
    load_checkpoint_hybrid(sv_model, weight_path)
    sv_model.eval()
    with torch.no_grad():
        sv_pred = sv_model(X_tensor).detach().cpu().numpy().flatten()
    sv_labels = predictions_to_labels(sv_pred)
    sv_acc = float(accuracy_score(y_test, sv_labels))
    sv_f1 = float(f1_score(y_test, sv_labels, pos_label=1))

    if iqm_backend is not None:
        iqm_accs = []
        iqm_f1s = []

        repeat_iter = range(N_REPEATS)
        if HAS_TQDM:
            repeat_iter = tqdm(repeat_iter, desc=f"IQM {ansatz_name}")

        for rep in repeat_iter:
            hw_model, hw_est = build_iqm_model(
                iqm_backend, ansatz_fn, EVAL_SHOTS, NUM_QUBITS, ANSATZ_DEPTH
            )
            load_checkpoint_connector(hw_model, weight_path)
            hw_model.eval()
            t0 = time.time()
            with torch.no_grad():
                hw_pred = hw_model(X_tensor).detach().cpu().numpy().flatten()
            _wall = time.time() - t0
            hw_labels = predictions_to_labels(hw_pred)
            acc = float(accuracy_score(y_test, hw_labels))
            f1 = float(f1_score(y_test, hw_labels, pos_label=1))
            iqm_accs.append(acc)
            iqm_f1s.append(f1)

            run_rows.append(
                {
                    "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                    "ansatz": ansatz_name,
                    "shots": EVAL_SHOTS,
                    "repeat_index": rep,
                    "accuracy": acc,
                    "f1": f1,
                    "weight_path": str(weight_path),
                    "test_csv": str(TEST_DATA_CSV),
                    "n_samples": n_samples,
                    "qpu_time_total": float(hw_est.total_qpu_time),
                    "wall_time_forward_s": float(_wall),
                }
            )

        mean_acc = float(np.mean(iqm_accs))
        mean_f1 = float(np.mean(iqm_f1s))
        std_acc = float(np.std(iqm_accs, ddof=1)) if N_REPEATS > 1 else 0.0
        std_f1 = float(np.std(iqm_f1s, ddof=1)) if N_REPEATS > 1 else 0.0
    else:
        mean_acc = mean_f1 = std_acc = std_f1 = float("nan")

    summary_rows.append(
        {
            "ansatz": ansatz_name,
            "statevector_accuracy": sv_acc,
            "statevector_f1": sv_f1,
            "statevector_std_accuracy": 0.0,
            "statevector_std_f1": 0.0,
            "iqm_mean_accuracy": mean_acc,
            "iqm_std_accuracy": std_acc,
            "iqm_mean_f1": mean_f1,
            "iqm_std_f1": std_f1,
            "eval_shots": EVAL_SHOTS if RUN_IQM_HARDWARE else float("nan"),
            "n_repeats": N_REPEATS if RUN_IQM_HARDWARE else 0,
            "test_csv": str(TEST_DATA_CSV),
            "weight_path": str(weight_path),
        }
    )

    print(f"Statevector  acc={sv_acc:.4f} f1={sv_f1:.4f}")
    if iqm_backend is not None:
        print(f"IQM mean±std acc={mean_acc:.4f}±{std_acc:.4f}  f1={mean_f1:.4f}±{std_f1:.4f}")
    else:
        print("IQM skipped (RUN_IQM_HARDWARE=False)")

df_runs = pd.DataFrame(run_rows)
df_summary = pd.DataFrame(summary_rows)

runs_csv = out_dir / "run_level_results.csv"
summary_csv = out_dir / "summary_comparison.csv"
df_runs.to_csv(runs_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

print(f"\nSaved: {runs_csv}")
print(f"Saved: {summary_csv}")
display(df_summary)